# Reproduce Numbers-page figures from the Divine Discourses data pack

Evidence label for everything below: **Verified (reproduction)**. Each check recomputes a figure
the site publishes, using only the downloadable pack (no site code, no corpus access).

Pack: `divinediscourses-data-v1.3.1.tar.gz`, from <https://divinediscourses.org/export>.
The underlying morphology is the Leeds Quranic Arabic Corpus v0.4 (GPL); see `NOTICE-LEEDS.txt` in the pack.
A check that fails means the pack and the page disagree, which is a bug to report, not a result.

In [ ]:
import io, tarfile, urllib.request, hashlib
import pandas as pd

VERSION = "1.3.1"
URL = f"https://divinediscourses.org/data/exports/divinediscourses-data-v{VERSION}.tar.gz"
LOCAL = f"data/exports/divinediscourses-data-v{VERSION}.tar.gz"   # when run from a clone of the repo

try:
    blob = open(LOCAL, "rb").read()
except FileNotFoundError:
    blob = urllib.request.urlopen(URL).read()
print("pack bytes:", len(blob), "sha256:", hashlib.sha256(blob).hexdigest()[:16], "...")

tar = tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz")
read = lambda name: pd.read_csv(tar.extractfile(name))
roots = read("root-frequencies.csv")
surahs = read("surah-stats.csv")
by_surah = read("root-surah-counts.csv")

## Check 1: rooted word-units (Numbers page, "rooted tokens")
Every rooted token belongs to exactly one root, so the per-root counts must sum to the number of rooted tokens.

In [ ]:
PUBLISHED_ROOTED_TOKENS = 49967      # data/numbers.json totals.rootedTokens
PUBLISHED_ROOTS = 1642               # distinct roots

assert roots["totalCount"].sum() == PUBLISHED_ROOTED_TOKENS, roots["totalCount"].sum()
assert len(roots) == PUBLISHED_ROOTS, len(roots)
print("OK: sum of per-root counts =", roots["totalCount"].sum(), "over", len(roots), "roots")

## Check 2: total word-units, two independent tables
Surah token counts and per-root counts come from different export tables; the rooted share must be consistent.

In [ ]:
PUBLISHED_TOKENS = 77429
assert surahs["tokenCount"].sum() == PUBLISHED_TOKENS, surahs["tokenCount"].sum()
assert surahs["verseCount"].sum() == 6236
rooted_share = PUBLISHED_ROOTED_TOKENS / PUBLISHED_TOKENS
print(f"OK: {PUBLISHED_TOKENS:,} word-units in 114 surahs; {rooted_share:.2%} carry a root")

## Check 3: the five most frequent roots (Numbers page, "top roots")

In [ ]:
PUBLISHED_TOP5 = [("Alh", 2851), ("qwl", 1722), ("kwn", 1390), ("rbb", 980), ("Amn", 879)]
top5 = list(roots.sort_values(["totalCount", "root"], ascending=[False, True]).head(5)[["root", "totalCount"]].itertuples(index=False, name=None))
assert top5 == PUBLISHED_TOP5, top5
print("OK:", top5)

## Check 4: a per-root total equals the sum of its per-surah counts
A cross-table consistency check on a different pair of exports.

In [ ]:
per_surah_sum = by_surah.groupby("root")["count"].sum()
merged = roots.set_index("root")["totalCount"].to_frame().join(per_surah_sum.rename("fromSurahs"))
bad = merged[merged["totalCount"] != merged["fromSurahs"]]
assert bad.empty, bad.head()
print("OK: all", len(merged), "roots agree between root-frequencies and root-surah-counts")

### What this does and does not show
It shows the published figures are internally consistent and recomputable from the pack.
It does not validate the Leeds annotation itself: every number inherits that corpus's root assignments.